In [5]:
import os # A library to connect to Operating system
from os.path import exists # This just checks is there a file or folder at this path
import torch # for all the pytorch operations
import torch.nn as nn # to build the neural networks or to initilaize the neural networks
from torch.nn.functional import log_softmax, pad # to use the pytorch log of softmax, padding functionalities
import math # For mathematical functions
import copy # this is a python module to make copies of the object
import time # python time module
from torch.optim.lr_scheduler import LambdaLR # This is to adjust the learning rate, scheduler changes the learning rate according to our function as the training goes along
import pandas as pd # this is a python's library for tables
import altair as alt # this is a python's library for charts
from torch.utils.data import DataLoader, Dataset # This is a utility for loading the data when training a model
import spacy # it is a nlp library. I can do: part of speech tagging, grammer parsing, ner, lemmas.
import GPUtil # it is a tiny utility library that tells what your GPU is doing
import warnings # controls warning messages -- used below to hide them all
from torch.utils.data.distributed import DistributedSampler # splits the dataset across multiple GPUs so each one trains on a different slice
import torch.distributed as dist # This is a plumbing that allows multiple training processes to talk to each other
import torch.multiprocessing as mp # This starts a new python process
from torch.nn.parallel import DistributedDataParallel as DDP # This is what makes multiple GPU training work parallely
from collections import Counter # This is from collections library a counter to count
from datasets import load_dataset # this is a data loader, data loading for training or finetuning.

warnings.filterwarnings("ignore")   
RUN_EXAMPLES = True


In [6]:
# Some convinience helper functions used throughout the notebook

def is_interactive_notebook():
    """ 
    True if the file is being run and not imported
    """
    return __name__ == "__main__"


def show_example(fn, args=[]):
    """Runs fn and returns its result but only in the notebook.
    """
    if __name__ == "__main__" and RUN_EXAMPLES:
        return fn(*args)


def execute_example(fn, args=[]):
    """
    Same as show example but it throws the result away.
    """
    if __name__ == "__main__" and RUN_EXAMPLES:
        fn(*args)


class DummyOptimizer(torch.optim.Optimizer):
    """ 
    Fake optimizer that updates nothing (for eval mode).
    """
    def __init__(self):
        self.param_groups = [{"lr": 0}]
        None

    def step(self):
        None

    def zero_grad(self, set_to_none=False):
        None


class DummyScheduler:
    """
    Fake scheduler that changes nothing (for eval model)
    """
    def step(self):
        None

### Background
The problem: an RNN reads word 1, then word 2, then word 3... Each step waits for the one before it. You can't use a GPU properly because there's nothing to do in parallel — that's "sequential computation".

Their fix: those three models (Extended Neural GPU, ByteNet, ConvS2S) replaced the RNN with a CNN. A convolution slides the same filter over every position independently, so all positions compute at once. No waiting.

Why the paper brings them up: to set up its own pitch. Convolutions only see a small window, so linking two far-apart words needs many stacked layers — the number of steps grows with distance (linearly for ConvS2S, log for ByteNet). Attention connects any two positions in one step, no matter how far apart.

So: RNN = parallel ✗, long-range ✓. CNN = parallel ✓, long-range ✗. Transformer = both ✓.

1. "grows in the distance between positions" — a conv filter only sees a few neighbours, so linking two far-apart words means stacking layers until the windows overlap.

2. "difficult to learn dependencies between distant positions" — the signal has to survive a long chain of layers. Gradients weaken, information gets diluted on the way.

3. "reduced to a constant number of operations" — attention compares every position with every other position directly, in one step. Distance stops mattering.

4. "at the cost of reduced effective resolution due to averaging" — this is the honest catch. Attention's output is a weighted average of all the value vectors. Averaging blurs. If a word needs to pay attention to two different things at once, one average smears them into mush.

"an effect we counteract with Multi-Head Attention" — so run 8 smaller attentions side by side instead of one big one. Each head averages separately, so one can track the subject while another tracks the verb. Then concatenate. Eight sharp averages instead of one blurry one.

**Self-attention = a sentence looking at itself.**

Normal attention has two sequences: the English decoder looks at the German encoder. Self-attention has **one** sequence — the words look at other words in the *same* sentence.

("intra" just means *within*. Same thing, older name.)

Why it helps:

```
"The animal didn't cross the street because it was tired."
```

To understand `it`, the model looks back at `animal`. Each word builds its meaning from the other words around it. That's "compute a representation of the sequence".

**The rest of the paragraph is just a reading list.** The authors are saying: *self-attention is not our invention — it already works well in reading comprehension, summarization, and so on.* Memory networks are another example, using attention over a stored memory instead of stepping word by word.

**Why they bother:** so the next line lands harder. Everyone before used attention as a *helper* bolted onto an RNN or CNN. The Transformer is the first model made of **only** attention, with the RNN thrown out completely. That's the claim they're setting up.

## Part 1: Model Architecture

In [7]:
class EncoderDecoder(nn.Module): # We are using nn.Module because we want to make this class a Pytorch model and not a Python object.
    """
    A standard Encoder-Decoder architecture. Base for this and many other models.
    """

    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def forward(self, src, tgt, src_mask, tgt_mask):
        "Take in and process masked src and target sequences."
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)

    def encocde(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

It builds nothing — it just **stores the five pieces** someone hands it. The real construction happens in `make_model`.

The five, in the order the data flows:

```
German ids ──src_embed──► vectors ──encoder──► memory
                                                  │
English ids ──tgt_embed──► vectors ──decoder──◄───┘
                                        │
                                    generator ──► word probabilities
```

| piece | job |
|---|---|
| `src_embed` | German word ids → vectors (+ position info) |
| `encoder` | read the whole German sentence → `memory` |
| `tgt_embed` | English word ids → vectors (+ position info) |
| `decoder` | use `memory` + what's written so far → vectors |
| `generator` | final vector → a probability for every English word |

Two things worth noticing:

**Why `generator` is separate from `decoder`.** The decoder outputs vectors of size 512, not word choices. The generator does the last step (512 → vocab size). Keeping it apart is why the loss code can reach it directly as `module.generator`.

**Why nothing is built here.** Passing the parts in rather than creating them inside means you can swap any piece — a different encoder, a different embedding — without touching this class. That's why the docstring says *"Base for this and many other models."*

Small note: `super(EncoderDecoder, self).__init__()` is old Python 2 style. Modern Python is just `super().__init__()` — identical behaviour, less typing.

In [8]:
class Generator(nn.Module):
    "Define standard linear + softmax generation step."

    def __init__(self, d_model, vocab): # d_model = 512 (vector size), vocab = how many words we know
        super().__init__()
        self.proj = nn.Linear(d_model, vocab)  # one big layer: 512 numbers in -> one score per word out

    def forward(self, x):  # x = decoder output, shape (batch, seq_len, 512)
        return log_softmax(self.proj(x), dim=-1) # proj -> raw scores; log_softmax -> log-probabilities
        # dim=-1 = last axis (the vocab), so scores sum to 1 per word position
        

## Encoder and Decoder Stacks

### Encoder
This is composed of a stack of N = 6 identical layers.

In [9]:
def clones(module, N):
    """Produce N identical layers"""
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [10]:
class Encoder(nn.Module):                      # the encoder = 6 identical layers stacked on top of each other
    "Core encoder is a stack of N layers"

    def __init__(self, layer, N):              # layer = one EncoderLayer to copy, N = how many (6 in the paper)
        super(Encoder, self).__init__()        # required pytorch setup
        self.layers = clones(layer, N)         # make N separate copies, each with its own weights
        self.norm = LayerNorm(layer.size)      # one final normalization; layer.size = 512

    def forward(self, x, mask):                # x = word vectors (batch, seq_len, 512), mask hides the padding
        "Pass the input (and mask) through each layer in turn."
        for layer in self.layers:              # walk the 6 layers in order
            x = layer(x, mask)                 # output of one becomes input of the next
        return self.norm(x)                    # normalize once at the very end -> this is the "memory"

In [11]:
class LayerNorm(nn.Module):
    "Construct a layernorm module (See citation for details)."

    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

In [12]:
# class SublayerConnection(nn.Module):
#     """
#     A residual connection followed by a layer norm.
#     Note for code simplicity the norm is first as opposed to last.
#     """

#     def __init__(self, size, dropout):
#         super(SublayerConnection, self).__init__()
#         self.norm = LayerNorm(size)
#         self.dropout = nn.Dropout(dropout)

#     def forward(self, x, sublayer):
#         "Apply residual connection to any sublayer with the same size."
#         return x + self.dropout(sublayer(self.norm(x)))

In [13]:
class LayerNorm(nn.Module):
    "Construct a layernorm module (See citation for details)"
    
    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)       
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2